# FMNs Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import spacy
from emoatlas import EmoScores
from emoatlas.resources import _valences
from scipy import stats
from tqdm import tqdm
from collections import Counter
import json
import networkx as nx
import re
import os
import gc  # For garbage collection
import psutil  # For memory monitoring
from SpreadPy.Models.models import BaseSpreading
from SpreadPy import IOUtils, AnalyticalUtils
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler, Normalizer
import sys
import seaborn as sns
# It is recommended to use the new seaborn objects interface
import seaborn.objects as so

# Add project root (parent of this scripts/ folder) to sys.path once
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Now import from ../src folder (qui ho creato delle funzioni utili)
from src import importer, fmtools, fmn_valence

In [ ]:
# you do not need to run these download commands every time. nltk.download()
# saves these data packages to your local file system in the nltk_data directory.
# Once they are downloaded, they are available for NLTK to use in any session.

'''nltk.download('wordnet')
nltk.download('omw-1.4') # for italian and multilingual support, if needed
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # for word tokenization
nltk.download('averaged_perceptron_tagger') # for POS tagging
'''


In [ ]:
from nltk import word_tokenize, pos_tag

# Environment and models settings

In [ ]:
# Set language once; everything else adapts
language = "italian"  # or "english"
env_file_path = "../env"

def load_env_vars(path: str) -> None:
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                s = line.strip()
                if s and not s.startswith("#"):
                    key, value = s.split("=", 1)
                    os.environ[key] = value
        print(f"Loaded SPACY_MODEL_PATH: {os.environ.get('SPACY_MODEL_PATH')}")
        print(f"Loaded NLTK_DATA: {os.environ.get('NLTK_DATA')}\n")
    else:
        print(f"Warning: {path} file not found. Using default paths.")

def ensure_nltk_path():
    nltk_data_dir = os.environ.get("NLTK_DATA")
    if nltk_data_dir:
        try:
            import nltk
            if nltk_data_dir not in nltk.data.path:
                nltk.data.path.insert(0, nltk_data_dir)
        except Exception as e:
            print(f"Warning: could not set NLTK_DATA path: {e}")

def init_spacy_and_emo(language: str):
    SPACY_DEFAULTS = {
        "english": "en_core_web_lg",
        "italian": "it_core_news_lg",
    }
    # env overrides default
    requested = os.environ.get("SPACY_MODEL_PATH") or SPACY_DEFAULTS.get(language, "en_core_web_lg")

    def _load(model_name_or_path: str):
        try:
            return spacy.load(model_name_or_path), model_name_or_path
        except Exception as e:
            return None, e

    nlp, ok = _load(requested)
    if nlp is None:
        fallback = SPACY_DEFAULTS.get(language, "en_core_web_lg")
        if isinstance(ok, Exception):
            print(f"spaCy load failed for '{requested}': {ok}\nFalling back to '{fallback}'")
        nlp, ok2 = _load(fallback)
        if nlp is None:
            raise RuntimeError(
                f"Failed to load spaCy model. Tried '{requested}' and fallback '{fallback}'. "
                f"Install with: python -m spacy download {fallback}"
            )
        requested = fallback

    emo = EmoScores(language=language, spacy_model=requested)
    return nlp, emo

# Execute setup
load_env_vars(env_file_path)
ensure_nltk_path()

nlp, emo = init_spacy_and_emo(language)

In [ ]:
pd.set_option('display.max_rows', 100)

# fmn settings

In [ ]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
exported_location = f"{base_output_path}/merged-no-filters"
formatted_data_dir = 'formatting'
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'fmn'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica','ansia','stress','depressione','statistica','informatica','fisica','scienza','prestazione','esame','voto','valutazione','scuola','competizione','università','calcolo','disciplina','materia']
# Dynamically extract project name from exported location and build name based on keywords
project_name = f"{exported_location.split('/')[-1]}{'_' + '_'.join(keywords) if keywords else ''}"

# Lemmatization setting
lemmatize = False

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'all_fmns'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'


files_output_dir = f'{base_output_path}/{project_name}/analysis'
figures_output_dir = f'{base_output_path}/{project_name}/figures'

# Ensure the files and figures output directories exist
os.makedirs(files_output_dir, exist_ok=True)
os.makedirs(figures_output_dir, exist_ok=True)

### Data pre-processing functions

In [ ]:
def get_wordnet_pos(treebank_tag):
    """Convert POS tag to a format recognized by WordNetLemmatizer."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default to noun

# Lemmatization function for English text using NLTK
def lemmatize_text(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags
    ]
    return ' '.join(lemmatized_tokens)

# Lemmatization for Italian text using spaCy (NLTK lemmatization is in English)
def lemmatize_text_it(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

## Data preparation

In [ ]:
def filter_associations_by_occurrences(
    df: pd.DataFrame,
    min_association_occurrences: int | None = None,
    word_col: str = "word",
    assoc_only: bool = True,
    keep_cue_rows: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filter by association pairs (cue_word, word) frequency.

    Keeps only rows where the association pair (cue_word, word) occurs at least
    min_association_occurrences across the dataset. Rows with index==0 (cue rows)
    are always kept if keep_cue_rows is True.

    Notes
    - Only association rows (index > 0) are filtered; cue rows are never part of
      (cue, word) pairs.
    - If min_association_occurrences is None or <= 1, the input is returned unchanged.
    """
    if min_association_occurrences is None or min_association_occurrences <= 1:
        if verbose:
            print("No filtering applied (min_association_occurrences is None or <= 1).")
        return df

    required_cols = {"cue_word", word_col, "index"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Expected columns {missing} in df")

    work = df.copy()

    # Association rows are those with index > 0
    assoc_mask = work["index"] > 0

    # Count pair frequencies on association rows only
    pair_counts = (
        work.loc[assoc_mask, ["cue_word", word_col]]
            .dropna(subset=["cue_word", word_col])
            .groupby(["cue_word", word_col], dropna=False)
            .size()
            .rename("pair_count")
            .reset_index()
    )

    # Merge counts back to all rows; cue rows (index==0) will have NaN counts
    work = work.merge(pair_counts, on=["cue_word", word_col], how="left")

    # Allowed only if pair_count >= threshold for association rows
    keep_assoc = work["pair_count"].fillna(0).astype(int) >= int(min_association_occurrences)

    # Build final keep mask
    keep_mask = pd.Series(False, index=work.index)
    if keep_cue_rows:
        keep_mask |= (work["index"] == 0)
    # Filter association rows by pair frequency
    keep_mask |= (assoc_mask & keep_assoc)

    out = work.loc[keep_mask].drop(columns=["pair_count"])

    if verbose:
        # Print top pairs for inspection
        top_n = 50
        sorted_pairs = (
            pair_counts.sort_values("pair_count", ascending=False)
                        .head(top_n)
        )
        total_pairs = int(pair_counts.shape[0])
        print(f"\nTop {min(top_n, total_pairs)} association pairs (of {total_pairs} total):")
        if not sorted_pairs.empty:
            print(sorted_pairs.to_string(index=False))
        else:
            print("(no association pairs found)")

        before_rows = len(df)
        after_rows = len(out)
        before_assoc_rows = int(assoc_mask.sum())
        after_assoc_rows = int((out["index"] > 0).sum())
        kept_unique_pairs = (
            out.loc[out["index"] > 0, ["cue_word", word_col]]
              .dropna()
              .drop_duplicates()
              .shape[0]
        )
        print(f"\nApplied pair frequency filter >= {min_association_occurrences} on (cue_word, {word_col}).")
        print(f"Rows: {before_rows} -> {after_rows} (removed {before_rows - after_rows})")
        print(f"Association rows: {before_assoc_rows} -> {after_assoc_rows} (removed {before_assoc_rows - after_assoc_rows})")
        print(f"Kept unique association pairs: {kept_unique_pairs}")

    return out

### Valence categorization and distribution

In [ ]:
# function to plot valence categories distrubution
def plot_valence_distribution(df):
    '''Plot the distribution of valence categories in the DataFrame.
    '''
    plt.figure(figsize=(6,5))

    # count the number of occurrences of each valence category
    # and return the sorted df
    counts = df['valence_cat'].value_counts().sort_index()
    tick_positions = range(len(counts)) 
    counts.plot(kind='bar')

    plt.title('Valence categories distribution')
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.xticks(tick_positions, labels=['-1 (Negative)', '0 (Neutral)', '+1 (Positive)'], rotation=0)

    plt.tight_layout()
    plt.show()

## Forma Mentis Networks functions

In [ ]:
def monitor_memory():
    """Monitor current memory usage"""
    process = psutil.Process(os.getpid())
    memory_mb = process.memory_info().rss / 1024 / 1024
    print(f"Current memory usage: {memory_mb:.2f} MB")
    return memory_mb


def extract_bfmn_structural_features(fmnts, save_path, emo=emo, batch_size=10, node_valence_maps=None, persist=True):
    """
    Extract, export and persist (if persist) per-participant network structural features.
    Writes one JSON and one .txt per FMN under output_dir/output_subdir.

    Parameters:
    fmnts (dict): {participant_id: FormaMentisNetwork}
    save_path (str): Base output directory
    emo (EmoScores): Initialized EmoScores object
    batch_size (int): Batch size
    node_valence_maps (dict or None): {participant_id: {node: valence_cat_int(-1,0,1)}}

    Returns:
    dict: {participant_id: structural_features_dict}
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    output_dir = save_path
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    def _sanitize_number(x):
        '''Sanitize number by converting to float and handling NaN/Inf
        '''
        try:
            if x is None:
                return None
            if isinstance(x, (float, int)):
                if isinstance(x, float) and (np.isnan(x) or np.isinf(x)):
                    return None
                return float(x)
        except Exception:
            return None
        return x

    def _valence_label(vcat):
        '''Map valence category to label'''
        if vcat == 1:
            return "positive"
        if vcat == -1:
            return "negative"
        return "neutral"

    # Create a list of (participant_id, fmnt) tuples for batching
    items = list(fmnts.items())
    total_items = len(items)
    print(f"Extracting features from {total_items} networks in batches of {batch_size}")
    # Initial memory usage
    monitor_memory()

    net_str_features_dict = {}

    for i in range(0, total_items, batch_size):
        batch_items = items[i:i+batch_size]
        print(f"\nProcessing batch {i//batch_size + 1}/{(total_items + batch_size - 1)//batch_size}")

        for fmn_title, fmnt in batch_items:
            if not hasattr(fmnt, "vertices") or len(fmnt.vertices) == 0:
                print(f"Skipping {fmn_title}: empty network.")
                continue

            # Build an undirected NX graph from the FMN
            G = nx.Graph()
            G.add_nodes_from(fmnt.vertices)
            G.add_edges_from(fmnt.edges)

            # Apply node valence attributes if provided (fallback neutral)
            node_val_map = {}
            if isinstance(node_valence_maps, dict):
                node_val_map = node_valence_maps.get(fmn_title, {}) or {}

            # Each node ends up with both a numeric (valence_cat) and string (valence)
            # attribute, enabling later feature calculations (counts, assortativity)
            # and readable outputs.
            for n in G.nodes():
                # Get the valence category for the node (default 0 if not found)
                vcat = node_val_map.get(n, 0)
                # Sanitize and assign the valence attributes for the node.
                # Both numeric and string attributes for valence are set.
                G.nodes[n]["valence_cat"] = int(vcat) if vcat is not None else 0
                G.nodes[n]["valence"] = _valence_label(G.nodes[n]["valence_cat"])

            # Basic counts
            n_nodes = G.number_of_nodes()
            n_edges = G.number_of_edges()
            n_pos = sum(1 for _, d in G.nodes(data=True) if d.get("valence_cat") == 1)
            n_neg = sum(1 for _, d in G.nodes(data=True) if d.get("valence_cat") == -1)

            # Degree stats (undirected graph)
            degrees = [d for _, d in G.degree()]
            min_degree = min(degrees) if degrees else 0
            max_degree = max(degrees) if degrees else 0

            # Local graph-level average clustering.
            # It is the mean of the local clustering coefficients of all the nodes in the graph.
            try:
                # Try the standard method first (more accurate)
                avg_clust = nx.average_clustering(G) if n_nodes > 1 else 0.0
            except Exception:
                try:
                    # Fallback to approximation (less accurate but more robust)
                    avg_clust = nx.algorithms.approximation.clustering_coefficient.average_clustering(G)
                except Exception:
                    avg_clust = None

            # Degree assortativity
            try:
                degree_assortativity = nx.degree_assortativity_coefficient(G)
            except Exception:
                degree_assortativity = None

            # Attribute assortativity for 'valence'
            try:
                valence_assortativity = nx.attribute_assortativity_coefficient(G, "valence")
            except Exception:
                valence_assortativity = None

            # Closeness centrality extremes
            try:
                closeness = nx.closeness_centrality(G)
                max_closeness = max(closeness.values()) if closeness else None
                min_closeness = min(closeness.values()) if closeness else None
            except Exception:
                max_closeness = None
                min_closeness = None

            # ASPL, diameter, nodes and edges of the largest connected component
            # LCC + path stats
            if n_nodes == 0:
                lcc_aspl = None
                lcc_diameter = None
                lcc_nodes = 0
                lcc_edges = 0
            elif n_nodes == 1:
                lcc_aspl = 0.0
                lcc_diameter = 0
                lcc_nodes = 1
                lcc_edges = 0
            else:
                try:
                    undirected_G = G.to_undirected() if G.is_directed() else G
                    if nx.is_connected(undirected_G):
                        H = undirected_G
                    else:
                        largest_cc = max(nx.connected_components(undirected_G), key=len)
                        H = undirected_G.subgraph(largest_cc).copy()
                    lcc_nodes = H.number_of_nodes()
                    lcc_edges = H.number_of_edges()
                    lcc_aspl = nx.average_shortest_path_length(H) if lcc_nodes > 1 else 0.0
                    lcc_diameter = nx.diameter(H) if lcc_nodes > 1 else 0
                except Exception:
                    lcc_aspl = None
                    lcc_diameter = None
                    lcc_nodes = H.number_of_nodes() if 'H' in locals() else 0
                    lcc_edges = H.number_of_edges() if 'H' in locals() else 0

            # Compute z-scores using emo module
            try:
                zscores = emo.zscores(fmnt)
            except Exception:
                zscores = {}

            # Compute local clustering
            try:
                local_clustering = nx.clustering(G)
            except Exception:
                local_clustering = {}

            # Compute hubs: top 5 concepts by highest degree
            try:
                # DegreeView is iterable of (node, degree) pairs; convert/sort directly
                degree_pairs = list(G.degree())
                sorted_degrees = sorted(degree_pairs, key=lambda x: x[1], reverse=True)[:5]
            except Exception:
                sorted_degrees = []

            structural_features = {
                "n_nodes": n_nodes,
                "n_edges": n_edges,
                "n_positive_nodes": n_pos,
                "n_negative_nodes": n_neg,
                "min_degree": _sanitize_number(min_degree),
                "max_degree": _sanitize_number(max_degree),
                "average_clustering": _sanitize_number(avg_clust),
                "local_clustering": {k: _sanitize_number(v) for k, v in local_clustering.items()},
                "degree_assortativity": _sanitize_number(degree_assortativity),
                "valence_assortativity": _sanitize_number(valence_assortativity),
                "max_closeness_centrality": _sanitize_number(max_closeness),
                "min_closeness_centrality": _sanitize_number(min_closeness),
                "lcc_nodes": _sanitize_number(lcc_nodes),
                "lcc_edges": _sanitize_number(lcc_edges),
                "lcc_average_shortest_path_length": _sanitize_number(lcc_aspl),
                "lcc_diameter": _sanitize_number(lcc_diameter),
                "zscores": {k: _sanitize_number(v) for k, v in zscores.items()},
                "top_5_hubs_by_degree": [{"node": n, "degree": _sanitize_number(d)} for n, d in sorted_degrees],
            }

            # Save structural features in JSON
            out_path_json = os.path.join(output_dir, f"{fmn_title}_structural_features.json")
            with open(out_path_json, "w", encoding="utf-8") as f:
                json.dump(structural_features, f, ensure_ascii=False, indent=2)
            print(f"Saved at {out_path_json}")

            # Save TXT dump
            out_path_txt = os.path.join(output_dir, f"{fmn_title}_structural_features.txt")
            with open(out_path_txt, "w", encoding="utf-8") as f:
                for k, v in structural_features.items():
                    f.write(f"{k}: {v}\n")
            print(f"Saved at {out_path_txt}")

            net_str_features_dict[fmn_title] = structural_features

        monitor_memory()
        gc.collect()

    print(f"Saved structural features for {len(net_str_features_dict)} networks\n")
    return net_str_features_dict


# An updated version of emo.export_whole_fmnt() that supports custom valence exports.
def export_whole_fmn_custom(fmnt, save_path, custom_valences):
    """
    Export a FMN with optional custom valence node lists.

    Parameters:
        fmnt : FormaMentisNetwork
        save_path : str
            Output .txt file path.
        custom_valences : list of sets
            [positive_set, negative_set, neutral_set]
            If None, uses lexicon-based valences from _valences().
    """
    if custom_valences:
        if not isinstance(custom_valences, (list, tuple)) or len(custom_valences) != 3:
            raise ValueError("custom_valences must be a list/tuple of three sets: [positive, negative, neutral]")
        pos_set, neg_set, neu_set = custom_valences
        if not all(isinstance(s, set) for s in (pos_set, neg_set, neu_set)):
            raise ValueError("custom_valences elements must be sets")
    else:
        # Get valence word sets for this language using lexicon
        positive_words, negative_words, _ = _valences(self.language)
    
    # Extract all nodes from fmnt
    all_nodes = set()
    edges = []

    for edge in fmnt.edges:
            if len(edge) >= 2:
                node1, node2 = edge[0], edge[1]
                edges.append((node1, node2))
                all_nodes.update([node1, node2])

    # Export only the nodes present in the FMN
    if custom_valences:
        # Categorize using custom sets; fall back to neutral if not in pos/neg
        positive_nodes = sorted([n for n in all_nodes if n in pos_set])
        negative_nodes = sorted([n for n in all_nodes if n in neg_set])
        neutral_nodes = sorted([
            n for n in all_nodes
            if (n in neu_set) or (n not in pos_set and n not in neg_set)
        ])

    # Write file
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        f.write("# Forma Mentis Network Export\n\n")
    
        f.write("## Edges\n")
        for n1, n2 in edges:
            f.write(f"{n1} -- {n2}\n")

        f.write("\n## Positive Nodes\n")
        for node in positive_nodes:
            f.write(f"{node}\n")

        f.write("\n## Negative Nodes\n")
        for node in negative_nodes:
            f.write(f"{node}\n")

        f.write("\n## Neutral Nodes\n")
        for node in neutral_nodes:
            f.write(f"{node}\n")

    print(f"[✔] Network exported successfully to '{save_path}'")


def export_whole_fmn_multiple_custom(fmnts, save_path, filename, custom_valences):
    """
    Bulk export FMNs with custom valences into txt files.

    Parameters:
        fmnts (dict or list): A dictionary or a list of forma mentis networks.
        save_path (str): The directory path to save the exported FMN txt files
        filename (str): Base filename to use if fmnts is a list
        fmn_type (str): A string to append to the filename (e.g., 'bfmn' or 'tfmn')
        custom_valences (list of sets): [positive_set, negative_set, neutral_set]
            Each set contains words corresponding to that valence category.
            If None, uses lexicon-based valences from _valences().
    """
    base_dir = os.path.join(save_path, fmn_type)
    if not os.path.exists(base_dir):
        os.makedirs(base_dir, exist_ok=True)

    all_nodes = set()
    edges = []

    if isinstance(fmnts, list):
        for item in fmnts:
            if len(item.edges) >= 2:
                node1, node2 = item.edges[0], item.edges[1]
                edges.append((node1, node2))
                all_nodes.update([node1, node2])
            # Export each FMN in the list
            out_path = os.path.join(base_dir, f"_{filename}.txt")
            export_whole_fmn_custom(item, out_path, custom_valences)

    elif isinstance(fmnts, dict):
        for title, fmnt in fmnts.items():
            # Export each FMN in the dict with its title
            out_path = os.path.join(base_dir, f"{title}_{fmn_type}.txt")
            export_whole_fmn_custom(fmnt, out_path, custom_valences)


def all_fmns_features_df(features_dict: dict, orient: str = "participants",
                         count_lists: bool = False,
                         list_suffix: str = "_count",
                         drop_nested_dicts: bool = False) -> pd.DataFrame:
    """
    Build a DataFrame from the features_dict returned by extract_fmn_structural_features.

    Parameters
    ----------
    features_dict : dict
        {participant_title: {feature_key: value}}
    orient : str
        "participants" -> rows=participants, columns=features (requested)
        "features"     -> rows=features, columns=participants (original behavior)
    count_lists : bool
        If True store length of list features (e.g. positive_nodes_count).
    list_suffix : str
        Suffix appended when counting list features.
    drop_nested_dicts : bool
        If True skip nested dicts (e.g. layer_metrics). Set False to flatten one level.
    """
    def _safe_col(name):
        # keep it readable but safe for CSV headers
        return str(name).replace(" ", "_")

    rows = []
    for participant, feat_dict in features_dict.items():
        row = {"participant_id": participant}
        for k, v in feat_dict.items():
            # Special flattening
            if k == "local_clustering" and isinstance(v, dict):
                for node, coeff in v.items():
                    row[f"local_clustering.{_safe_col(node)}"] = coeff
                continue
            if k == "zscores" and isinstance(v, dict):
                for emo_k, z in v.items():
                    row[f"zscores.{_safe_col(emo_k)}"] = z
                continue
            if k == "top_5_hubs_by_degree" and isinstance(v, list):
                for i, item in enumerate(v, start=1):
                    if isinstance(item, dict):
                        row[f"hub_{i}_node"] = item.get("node")
                        row[f"hub_{i}_degree"] = item.get("degree")
                    elif isinstance(item, (list, tuple)) and len(item) >= 2:
                        row[f"hub_{i}_node"] = item[0]
                        row[f"hub_{i}_degree"] = item[1]
                # If fewer than 5 hubs, ensure consistent columns
                for i in range(len(v) + 1, 6):
                    row.setdefault(f"hub_{i}_node", None)
                    row.setdefault(f"hub_{i}_degree", None)
                continue

            # Generic handling
            if isinstance(v, dict):
                if drop_nested_dicts:
                    continue
                for sk, sv in v.items():
                    if isinstance(sv, (list, dict)):
                        continue
                    row[f"{k}.{_safe_col(sk)}"] = sv
            elif isinstance(v, list):
                if count_lists:
                    row[f"{k}{list_suffix}"] = len(v)
                else:
                    # Store as JSON-like string to keep content
                    try:
                        row[k] = json.dumps(v, ensure_ascii=False)
                    except Exception:
                        row[k] = str(v)
            else:
                row[k] = v
        rows.append(row)

    if orient == "participants":
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.set_index("participant_id").sort_index()
        return df

    # Original wide style (features as rows)
    series_list = []
    for r in rows:
        participant_id = r.pop("participant_id")
        series_list.append(pd.Series(r, name=participant_id))
    if not series_list:
        return pd.DataFrame()
    wide_df = pd.concat(series_list, axis=1)
    return wide_df


def inspect_fmn(fmn, participant = str, all_features = dict, emo=emo):
    '''Inspection of a single FMN (also multi-layer). Converts to NetworkX, prints sizes and basic checks.
       Works whether fmnt.edges is a dict (e.g., {'syntactic': [...], 'synonyms': [...]})
       or a flat iterable of 2-tuples.

    Parameters:
        fmn : FormaMentisNetwork (also multi-layer)
        participant : str
            Participant identifier for logging.
        all_features : dict
            {participant: features_dict} for printing features.
        emo : EmoScores
    '''
    def _collect_edges(edges_attr):
        # Normalize edges from different fmnt formats into a list of (u, v)
        out = []
        if edges_attr is None:
            return out
        if isinstance(edges_attr, dict):
            for lst in edges_attr.values():
                for e in lst:
                    if isinstance(e, (list, tuple)) and len(e) >= 2:
                        out.append((e[0], e[1]))
        else:
            for e in edges_attr:
                if isinstance(e, (list, tuple)) and len(e) >= 2:
                    out.append((e[0], e[1]))
        return out

    print(f"FMN: {participant}")

    # Before conversion: get vertices/edges if available
    n_vertices = len(getattr(fmn, "vertices", [])) if hasattr(fmn, "vertices") else None
    fmnt_edges = _collect_edges(getattr(fmn, "edges", None))
    fmnt_edges = list(set(tuple(map(str, e)) for e in fmnt_edges))  # dedup; ensure strings
    print("Before conversion to nx graph:")
    if n_vertices is not None:
        print(f"{n_vertices} nodes and {len(fmnt_edges)} edges")
    else:
        print(f"(vertices not available) edges: {len(fmnt_edges)}")

    # Convert to NetworkX
    fmn_G = emo.formamentis_to_nxgraph(fmn)
    print("\nAfter conversion to nx graph:")
    print(fmn_G)

    # Basic consistency check (only if we could read vertices)
    if n_vertices is not None and fmn_G.number_of_nodes() != n_vertices:
        print(f"Warning: Node count mismatch after conversion! for {participant}")

    # Compare edges (robust to undirected orientation)
    missing_edges = []
    if fmnt_edges:
        G_edges = set()
        if isinstance(fmn_G, (nx.Graph, nx.DiGraph)):
            if fmn_G.is_directed():
                G_edges = set((str(u), str(v)) for u, v in fmn_G.edges())
            else:
                G_edges = set(tuple(sorted((str(u), str(v)))) for u, v in fmn_G.edges())
        for u, v in fmnt_edges:
            e = (str(u), str(v))
            if fmn_G.is_directed():
                if e not in G_edges:
                    missing_edges.append((u, v))
            else:
                if tuple(sorted(e)) not in G_edges:
                    missing_edges.append((u, v))
    if missing_edges:
        print(f"Warning: {len(missing_edges)} edges from FMN not found in NetworkX graph for {participant}: {missing_edges[:20]}{' ...' if len(missing_edges) > 20 else ''}")

    # Print provided features (if any)
    print("\n--------------------------------------------")
    print(f"Features for {participant}:")
    print("--------------------------------------------")
    features_for_participant = (all_features or {}).get(participant, {})
    if isinstance(features_for_participant, dict):
        for k, v in features_for_participant.items():
            print(k, v)
    else:
        print(features_for_participant)
    print("\n---\n")


def draw_fmnts(
    fmnts,
    out_dir,
    layout,
    highlight,
    thickness,
    ax=None,
    emo=emo,
    custom_valences=None,
    filename_base=None,
    batch_size=5,
    figsize=(10, 10),
    dpi=300,
    bbox_pad=0.08,
    margin=0.05,
    fix_center_mask=True,
    bring_edges_to_front=True,
):
    """
    Draw FMNs and save images directly in out_dir (no subfolders).
    If filename_base is given, it is used; otherwise <fmnt_title>.

    Notes
    - For edge_bundling/circle layout, EmoAtlas draws a central white patch to hide the hub.
      Set fix_center_mask=True to make that patch transparent and avoid "missing" edges near the center.
    - bbox_inches='tight' + pad_inches keeps labels/curves fully in-frame.
    """
    from matplotlib.patches import Circle

    os.makedirs(out_dir, exist_ok=True)

    items = list(fmnts.items())
    total_items = len(items)
    print(f"Drawing {total_items} networks in batches of {batch_size}")

    for i in range(0, total_items, batch_size):
        batch_items = items[i:i + batch_size]
        print(f"\nDrawing batch {i // batch_size + 1}/{(total_items + batch_size - 1) // batch_size}")
        monitor_memory()

        for fmnt_title, fmnt in batch_items:
            if not hasattr(fmnt, "vertices") or len(fmnt.vertices) == 0:
                print(f"Skipping {fmnt_title}: the network has zero vertices.")
                continue

            print(f"Drawing TFMN: {fmnt_title}")

            # Create a figure only if caller didn't pass one
            created_fig = False
            if ax is None:
                fig, ax_ = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
                created_fig = True
            else:
                ax_ = ax
                fig = ax_.figure

            # Draw with EmoAtlas
            emo.draw_formamentis(
                fmn=fmnt,
                layout=layout,
                highlight=highlight,
                thickness=thickness,
                ax=ax_,
                save_path=None,  # prevent internal saving
                custom_valences=custom_valences,
            )

            # Optional: neutralize the central white mask that hides bundled arcs
            if fix_center_mask or bring_edges_to_front:
                try:
                    # Find smallest circle patch (the hub used by the circle layout)
                    circles = [p for p in ax_.patches if isinstance(p, Circle) and hasattr(p, "radius")]
                    if fix_center_mask and circles:
                        hub = min(circles, key=lambda c: getattr(c, "radius", float("inf")))
                        # Make it fully transparent
                        hub.set_facecolor((1, 1, 1, 0))
                        hub.set_edgecolor((1, 1, 1, 0))
                        hub.set_alpha(0.0)
                        hub.set_visible(True)
                        # Push it to the back
                        hub.set_zorder(0)

                    if bring_edges_to_front:
                        # Lines and curves live in collections
                        for coll in ax_.collections:
                            coll.set_zorder(20)
                        # Keep labels on top
                        for txt in ax_.texts:
                            txt.set_zorder(30)
                        # Other patches (outer rings, etc.) under the edges
                        for p in ax_.patches:
                            if not isinstance(p, Circle):
                                p.set_zorder(10)
                except Exception as e:
                    print(f"[WARN] Could not adjust center mask/zorder: {e}")

            # Ensure full content fits
            ax_.set_aspect("equal", adjustable="datalim")
            ax_.relim()
            ax_.autoscale_view()
            ax_.margins(x=margin, y=margin)
            ax_.axis("off")

            base = filename_base if filename_base else fmnt_title
            png_path = os.path.join(out_dir, f"{base}.png")
            svg_path = os.path.join(out_dir, f"{base}.svg")

            # Save tightly with padding to include outer labels/curves
            fig.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=bbox_pad, facecolor="white")
            fig.savefig(svg_path, bbox_inches="tight", pad_inches=bbox_pad, facecolor="white")
            print(f"Saved {png_path}")
            print(f"Saved {svg_path}")

            if created_fig:
                plt.close(fig)

        gc.collect()

### Data loading and pre-processing

In [ ]:
print(f"Importing fmnts from project: {exported_location}")

fmn_names = [
    "highschool_north_high_anxiety_bfmns",
    "highschool_north_low_anxiety_bfmns",
    "highschool_south_filtered_bfmns",
    "highschool_south_unfiltered_bfmns",
    "high_school_low_anxiety_gpt_oss_bfmns",
    "highschool_high_anxiety_gpt_oss_bfmns",
    "psychology_low_anxiety_gpt_oss_bfmns",
    "psychology_high_anxiety_gpt_oss_bfmns",
    "psychology_high_anxiety_bfmns",
    "psychology_low_anxiety_bfmns",
    "physics_bfmns",
    "physics_gpt_oss_bfmns",
    "researchers_filtered_bfmns",
    "researchers_unfiltered_bfmns",
]

# Construct the full paths to the .txt files, removing _bfmns or _tfmns suffixes
if methods == {'individual'}:
    fmn_paths = [
        os.path.join(exported_location, f"individual/{name}/{name.replace('_bfmns', '').replace('_tfmns', '')}.txt") for name in fmn_names
    ]
elif methods == {'aggregated'}:
    fmn_paths = [
        os.path.join(exported_location, f"aggregated/{name}/{name.replace('_bfmns', '').replace('_tfmns', '')}.txt") for name in fmn_names
    ]

print("Importing FMNs from the following paths:")
for path in fmn_paths:
    print(f" - {path}")

fmns = {}
# Additional dictionary to hold custom valences per fmn (lexicon-based when custom not available)
valences_per_fmn = {}
for path in fmn_paths:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Expected FMN file not found: {path}")
    print(path)

    try:
        imported = importer.import_fmnt_from_txt(
            filepath=path, language=language, custom_valences=True
        )
        # Store custom valences for this fmn
        valences_per_fmn[os.path.basename(os.path.dirname(path))] = imported.custom_valences
    except Exception as e:
        print(f"[WARN] Could not import with custom valences: {e}")
        imported = importer.import_fmnt_from_txt(
            filepath=path, language=language, custom_valences=False
        )
        # Store custom valences for this fmn (as lexicon-based)
        valences_per_fmn[os.path.basename(os.path.dirname(path))] = imported.custom_valences

    # Use folder name as key; store the FormamentisNetwork object
    title = os.path.basename(os.path.dirname(path))
    fmns[title] = imported.fmnt

## Run the pipeline on preprocessed data

### Aggregated pipeline

In [ ]:
# Method to execute: a single aggregated BFMN from all data
if 'aggregated' in methods:
    # Monitor initial memory usage
    print("Starting aggregated pipeline...\n")

    # Verify presence of keywords in each FMN
    fmtools.verify_and_report_nodes(
        fmns=fmns,
        required=keywords,
        strict=True
    )

    monitor_memory()

    for group_name, fmn in fmns.items():
        # fmn custom valences extraction for the fmn
        custom_valences = valences_per_fmn.get(group_name, None)
        valence_stats = fmn_valence.fmn_valence_stats(fmn, custom_valences)

        # Inspect the FMN
        inspect_fmn(
            fmn=fmn,
            participant=group_name,
            all_features={group_name: valence_stats},
            emo=emo
        )

# Deep patterns analysis

In [ ]:
from collections import defaultdict

def compare_fmns_patterns(
    fmns: dict,
    language: str = "italian",
    custom_valences=None,
    top_k: int = 20,
    min_edge_freq: int | None = None,
):
    """
    Compare all FMNs' nodes and edges with valences to highlight strongest recurring patterns.

    Parameters
    - fmns: dict[str, FormamentisNetwork]
    - language: 'italian' | 'english' (used when custom_valences is None)
    - custom_valences: list/tuple of 3 sets [positive, negative, neutral] or None (lexicon fallback)
    - top_k: number of top nodes/edges shown in the printed summary
    - min_edge_freq: optional threshold to filter edges by frequency (in how many FMNs they appear)

    Returns
    dict with:
      - nodes_df: per-node presence across FMNs + valence breakdown and dominance
      - edges_df: per-edge presence across FMNs + valence pair type
      - edge_type_dist_df: distribution of edge valence-pair types (counts and shares)
      - core: dict with 'nodes_all_fmns' and 'edges_ge_half' sets
      - node_valence_maps: per-FMN mapping {fmn_title: {node: -1|0|1}}
    """
    # Resolve valence sets
    if custom_valences is not None:
        if not (isinstance(custom_valences, (list, tuple)) and len(custom_valences) == 3 and all(isinstance(s, set) for s in custom_valences)):
            raise ValueError("custom_valences must be a list/tuple of three sets: [positive, negative, neutral]")
        pos_set, neg_set, neu_set = custom_valences
    else:
        pos_set, neg_set, amb_set = _valences(language)
        neu_set = set(amb_set)  # treat ambivalent as neutral

    def val_cat(n: str) -> int:
        if n in pos_set: return 1
        if n in neg_set: return -1
        if n in neu_set: return 0
        return 0  # fallback neutral

    def pair_label(v1: int, v2: int) -> str:
        a, b = sorted((v1, v2))
        lab = {
            (-1, -1): "neg-neg",
            (-1, 0):  "neg-neu",
            (-1, 1):  "pos-neg",
            (0, 0):   "neu-neu",
            (0, 1):   "pos-neu",
            (1, 1):   "pos-pos",
        }
        return lab[(a, b)]

    def collect_edges(edges_attr):
        """Normalize edges from different fmnt formats into a set of undirected tuples (u, v)."""
        out = set()
        if edges_attr is None:
            return out
        if isinstance(edges_attr, dict):
            for lst in edges_attr.values():
                for e in lst:
                    if isinstance(e, (list, tuple)) and len(e) >= 2 and e[0] != e[1]:
                        out.add(tuple(sorted((str(e[0]), str(e[1])))))
        else:
            for e in edges_attr:
                if isinstance(e, (list, tuple)) and len(e) >= 2 and e[0] != e[1]:
                    out.add(tuple(sorted((str(e[0]), str(e[1])))))
        return out

    total_n = len(fmns)
    node_presence = Counter()                       # across FMNs
    node_valence_counts = defaultdict(Counter)      # node -> counts of {-1,0,1} across FMNs
    node_valence_maps = {}                          # per FMN valence map

    edge_presence = Counter()                       # (u,v) across FMNs
    edge_valence_type_counts = Counter()            # pair-type label counts weighted by FMN presence
    edge_valence_pair = {}                          # per edge -> label

    # Per-FMN loop
    for title, fmn in fmns.items():
        vertices = set(map(str, getattr(fmn, "vertices", [])))
        edges = collect_edges(getattr(fmn, "edges", None))

        # Node valence map for this FMN
        vmap = {n: val_cat(n) for n in vertices}
        node_valence_maps[title] = vmap

        # Node presence and valence tallies
        for n in vertices:
            node_presence[n] += 1
            node_valence_counts[n][vmap[n]] += 1

        # Edge presence and valence-pair type tallies
        for (u, v) in edges:
            edge_presence[(u, v)] += 1
            v1, v2 = vmap.get(u, 0), vmap.get(v, 0)
            label = pair_label(v1, v2)
            edge_valence_pair[(u, v)] = label

    # Build nodes_df
    rows_nodes = []
    for n, cnt in node_presence.items():
        vc = node_valence_counts[n]
        pos_c, neg_c, neu_c = vc.get(1, 0), vc.get(-1, 0), vc.get(0, 0)
        dominant_val = max(vc.items(), key=lambda kv: (kv[1], kv[0]))[0] if vc else 0
        rows_nodes.append({
            "node": n,
            "n_fmns": cnt,
            "share": cnt / total_n if total_n else 0,
            "pos_count": pos_c,
            "neg_count": neg_c,
            "neu_count": neu_c,
            "dominant_valence": dominant_val,  # 1 pos, 0 neu, -1 neg
            "dominance_ratio": (vc[dominant_val] / cnt) if cnt else 0.0,
        })
    nodes_df = pd.DataFrame(rows_nodes).sort_values(
        ["n_fmns", "dominance_ratio", "node"], ascending=[False, False, True]
    ).reset_index(drop=True)

    # Build edges_df
    # Optionally filter by min_edge_freq
    edges_iter = ((e, c) for e, c in edge_presence.items()
                  if (min_edge_freq is None or c >= min_edge_freq))
    rows_edges = []
    type_counter = Counter()
    for (u, v), cnt in edges_iter:
        label = edge_valence_pair.get((u, v), "unknown")
        type_counter[label] += cnt
        rows_edges.append({
            "u": u,
            "v": v,
            "edge": f"{u} -- {v}",
            "n_fmns": cnt,
            "share": cnt / total_n if total_n else 0,
            "valence_pair_type": label,
        })
    edges_df = pd.DataFrame(rows_edges).sort_values(
        ["n_fmns", "edge"], ascending=[False, True]
    ).reset_index(drop=True)

    # Edge type distribution (weighted by presence across FMNs)
    if type_counter:
        total_edge_occ = sum(type_counter.values())
        edge_type_dist_df = (
            pd.DataFrame(
                [{"pair_type": t, "count": c, "share": c / total_edge_occ}
                 for t, c in type_counter.items()]
            ).sort_values(["count", "pair_type"], ascending=[False, True]).reset_index(drop=True)
        )
    else:
        edge_type_dist_df = pd.DataFrame(columns=["pair_type", "count", "share"])

    # Core sets
    core_nodes = set(n for n, c in node_presence.items() if c == total_n)
    ge_half = int(np.ceil(total_n * 0.5))
    core_edges_ge_half = set(e for e, c in edge_presence.items() if c >= ge_half)

    # Print concise summary
    print(f"\nCompared {total_n} FMNs")
    print(f"- Unique nodes: {len(node_presence)} | nodes in ALL FMNs: {len(core_nodes)}")
    print(f"- Unique edges: {len(edge_presence)} | edges in >=50% FMNs: {len(core_edges_ge_half)}")
    if not nodes_df.empty:
        print(f"\nTop {min(top_k, len(nodes_df))} nodes by presence:")
        display(nodes_df.head(top_k))
    if not edges_df.empty:
        print(f"\nTop {min(top_k, len(edges_df))} edges by presence:")
        display(edges_df.head(top_k))
    if not edge_type_dist_df.empty:
        print("\nValence-pair edge type distribution (weighted by FMN presence):")
        display(edge_type_dist_df)

    return {
        "nodes_df": nodes_df,
        "edges_df": edges_df,
        "edge_type_dist_df": edge_type_dist_df,
        "core": {
            "nodes_all_fmns": core_nodes,
            "edges_ge_half": core_edges_ge_half,
        },
        "node_valence_maps": node_valence_maps,
    }

results = compare_fmns_patterns(
    fmns=fmns,
    language=language,
    custom_valences=globals().get("custom_valences"),  # or None to use lexicon
    top_k=25,
    min_edge_freq=2,  # show edges appearing in at least 2 FMNs
)

In [ ]:
def compute_node_aura(
    fmn,
    target_word: str,
    emo=emo,
    language: str | None = None,
    custom_valences=None,
    include_target: bool = False,
    tie_priority: tuple = (0, 1, -1),  # tie-break preference: neutral > positive > negative
    return_details: bool = True,
    case_insensitive: bool = True,
):
    """
    Compute the aura (most frequent sentiment polarity) in the semantic frame of target_word.

    Parameters
    - fmn: FormamentisNetwork
    - target_word: node label to extract the semantic frame for
    - emo: EmoScores instance (used to extract the frame)
    - language: optional, defaults to emo.language if None
    - custom_valences: [pos_set, neg_set, neu_set] or None to use lexicon (_valences)
    - include_target: include the center node in the polarity tally
    - tie_priority: tuple of valence categories for tie-breaking, default (0, 1, -1)
    - return_details: if True, include per-category node lists
    - case_insensitive: match target node and valence sets case-insensitively

    Returns
    dict with:
      node, aura_cat (-1/0/1), aura_label, counts, shares, frame_size, considered_size,
      considered_nodes, [optional] pos_nodes/neg_nodes/neu_nodes
    """
    if language is None:
        language = getattr(emo, "language", "italian")

    # Resolve valence sets
    if custom_valences is not None:
        if not (isinstance(custom_valences, (list, tuple)) and len(custom_valences) == 3 and all(isinstance(s, set) for s in custom_valences)):
            raise ValueError("custom_valences must be [positive_set, negative_set, neutral_set]")
        pos_set, neg_set, neu_set = custom_valences
    else:
        pos_set, neg_set, amb_set = _valences(language)
        neu_set = set(amb_set)

    # Normalize helper
    def norm(s: str) -> str:
        return str(s).casefold() if case_insensitive else str(s)

    pos_l = {norm(w) for w in pos_set}
    neg_l = {norm(w) for w in neg_set}
    neu_l = {norm(w) for w in neu_set}

    # Find actual node label in FMN (preserve original casing for reporting)
    vertices = list(getattr(fmn, "vertices", []))
    vmap_lower = {norm(v): v for v in vertices}
    target_norm = norm(target_word)
    if target_norm not in vmap_lower:
        raise ValueError(f"Node '{target_word}' not found in FMN vertices.")
    target_actual = vmap_lower[target_norm]

    # Extract the semantic frame FMN around target
    frame_fmn = emo.extract_word_from_formamentis(fmn, target_actual)
    frame_vertices = list(getattr(frame_fmn, "vertices", []))

    # Build considered node list
    considered = []
    for n in frame_vertices:
        if not include_target and norm(n) == target_norm:
            continue
        considered.append(n)

    # Tally polarities
    counts = Counter({1: 0, 0: 0, -1: 0})
    pos_nodes, neg_nodes, neu_nodes = [], [], []

    for n in considered:
        nl = norm(n)
        if nl in pos_l:
            counts[1] += 1
            pos_nodes.append(n)
        elif nl in neg_l:
            counts[-1] += 1
            neg_nodes.append(n)
        else:
            # explicit neutral set or fallback neutral
            counts[0] += 1
            neu_nodes.append(n)

    total = max(1, len(considered))
    max_count = max(counts.values()) if total > 0 else 0
    # Tie-break using tie_priority order
    top_cats = [cat for cat, c in counts.items() if c == max_count]
    aura_cat = min(top_cats, key=lambda c: tie_priority.index(c)) if top_cats else 0
    label = {1: "positive", 0: "neutral", -1: "negative"}[aura_cat]

    result = {
        "node": target_actual,
        "aura_cat": aura_cat,
        "aura_label": label,
        "counts": {
            "positive": counts[1],
            "neutral": counts[0],
            "negative": counts[-1],
        },
        "shares": {
            "positive": counts[1] / total,
            "neutral": counts[0] / total,
            "negative": counts[-1] / total,
        },
        "frame_size": len(frame_vertices),
        "considered_size": len(considered),
        "considered_nodes": considered,
    }
    if return_details:
        result.update({
            "pos_nodes": pos_nodes,
            "neg_nodes": neg_nodes,
            "neu_nodes": neu_nodes,
        })
    return result


def compute_auras_for_words(
    fmn,
    words: list[str],
    emo=emo,
    language: str | None = None,
    custom_valences=None,
    include_target: bool = False,
    case_insensitive: bool = True,
):
    """
    Batch aura computation for a list of words. Returns DataFrame.
    """
    rows = []
    for w in words:
        try:
            r = compute_node_aura(
                fmn=fmn,
                target_word=w,
                emo=emo,
                language=language,
                custom_valences=custom_valences,
                include_target=include_target,
                return_details=False,
                case_insensitive=case_insensitive,
            )
            rows.append({
                "node": r["node"],
                "aura_cat": r["aura_cat"],
                "aura_label": r["aura_label"],
                "pos_count": r["counts"]["positive"],
                "neu_count": r["counts"]["neutral"],
                "neg_count": r["counts"]["negative"],
                "pos_share": r["shares"]["positive"],
                "neu_share": r["shares"]["neutral"],
                "neg_share": r["shares"]["negative"],
                "frame_size": r["frame_size"],
                "considered_size": r["considered_size"],
            })
        except Exception as e:
            rows.append({
                "node": w, "aura_cat": None, "aura_label": None,
                "pos_count": None, "neu_count": None, "neg_count": None,
                "pos_share": None, "neu_share": None, "neg_share": None,
                "frame_size": 0, "considered_size": 0, "error": str(e)
            })
    return pd.DataFrame(rows).sort_values(["aura_cat", "node"], ascending=[False, True]).reset_index(drop=True)

# Batch on your keywords for the first FMN
auras_df = compute_auras_for_words(list(fmns.values())[0], keywords, emo=emo,
                                   custom_valences=globals().get("custom_valences"))
display(auras_df.head(20))

In [ ]:
def keyword_correlometer_bulk(
    fmns: dict,
    keywords: list[str],
    emo=emo,
    per_fmn_custom_valences: dict | None = None,
    include_target: bool = False,
    language: str | None = None,
    annotate: bool = True,
    figsize=(8.0, 6.5),
    dpi=300,
):
    """
    Compute 'correlometer' (importance vs valence) for all keywords across all FMNs,
    and plot all points in a single figure. Legends are placed to avoid covering the plot:
      - Valence legend stays small inside the axes (lower right)
      - FMN group legend is moved outside at the bottom, in multiple columns
    """
    import math
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    if language is None:
        language = getattr(emo, "language", "italian")

    # Reuse single-node function for aura
    def valence_for_node(fmn, node_label, custom_sets):
        return compute_node_aura(
            fmn=fmn,
            target_word=node_label,
            emo=emo,
            language=language,
            custom_valences=custom_sets,
            include_target=include_target,
            return_details=False,
            case_insensitive=False,
        )

    # Min-max scaler for dicts
    def minmax_scale(d):
        vals = list(d.values())
        if not vals:
            return {n: 0.0 for n in d}
        vmin, vmax = min(vals), max(vals)
        if vmax <= vmin:
            return {n: 0.0 for n in d}
        return {n: (v - vmin) / (vmax - vmin) for n, v in d.items()}

    rows = []
    missing = []

    # Pre-assign a marker per FMN to distinguish groups
    marker_cycle = ["o", "s", "D", "^", "v", "P", "X", "*", "<", ">", "h", "8"]
    group_names = list(fmns.keys())
    marker_map = {g: marker_cycle[i % len(marker_cycle)] for i, g in enumerate(group_names)}

    # Color by aura
    color_map = {"positive": "#1f77b4", "neutral": "#7f7f7f", "negative": "#d62728"}

    # Precompute centralities per FMN once
    centralities_cache = {}

    for group, fmn in fmns.items():
        # Build NX Graph and centralities (cache)
        if group not in centralities_cache:
            G = emo.formamentis_to_nxgraph(fmn)
            # Robust centralities
            try:
                deg_c = nx.degree_centrality(G)
            except Exception:
                deg_c = {n: 0.0 for n in G.nodes()}
            try:
                bet_c = nx.betweenness_centrality(G, normalized=True)
            except Exception:
                bet_c = {n: 0.0 for n in G.nodes()}
            try:
                clo_c = nx.closeness_centrality(G)
            except Exception:
                clo_c = {n: 0.0 for n in G.nodes()}
            try:
                eig_c = nx.eigenvector_centrality(G, max_iter=1000)
            except Exception:
                try:
                    eig_c = nx.pagerank(G, alpha=0.85)
                except Exception:
                    eig_c = {n: 0.0 for n in G.nodes()}

            deg_s = minmax_scale(deg_c)
            bet_s = minmax_scale(bet_c)
            clo_s = minmax_scale(clo_c)
            eig_s = minmax_scale(eig_c)

            centralities_cache[group] = {
                "G": G,
                "deg_s": deg_s, "bet_s": bet_s, "clo_s": clo_s, "eig_s": eig_s
            }

        G = centralities_cache[group]["G"]
        deg_s = centralities_cache[group]["deg_s"]
        bet_s = centralities_cache[group]["bet_s"]
        clo_s = centralities_cache[group]["clo_s"]
        eig_s = centralities_cache[group]["eig_s"]

        # Resolve custom valences for this FMN if provided
        custom_sets = None
        if isinstance(per_fmn_custom_valences, dict):
            custom_sets = per_fmn_custom_valences.get(group, None)

        # Case-insensitive lookup of nodes in this FMN
        vertices = list(getattr(fmn, "vertices", G.nodes()))
        def norm(s): return str(s).casefold()
        vmap_lower = {norm(v): v for v in vertices}

        for kw in keywords:
            kn = norm(kw)
            if kn not in vmap_lower:
                missing.append((group, kw))
                continue
            node = vmap_lower[kn]

            # Importance = mean of scaled centralities
            components = {
                "degree": deg_s.get(node, 0.0),
                "betweenness": bet_s.get(node, 0.0),
                "closeness": clo_s.get(node, 0.0),
                "eigen_or_pagerank": eig_s.get(node, 0.0),
            }
            importance = float(np.mean(list(components.values()))) if components else 0.0

            # Valence via aura of node's frame
            aura = valence_for_node(fmn, node, custom_sets)
            valence_score = aura["shares"]["positive"] - aura["shares"]["negative"]

            rows.append({
                "group": group,
                "keyword": kw,
                "node": node,
                "degree": int(G.degree(node)) if node in G else 0,
                "importance": importance,
                "aura_label": aura["aura_label"],
                "aura_cat": aura["aura_cat"],
                "valence_score": float(valence_score),
                "pos_share": aura["shares"]["positive"],
                "neg_share": aura["shares"]["negative"],
                "neu_share": aura["shares"]["neutral"],
                "n_neighbors": int(aura["considered_size"]),
                "marker": marker_map[group],
                "color": color_map.get(aura["aura_label"], "#7f7f7f"),
            })

    df = pd.DataFrame(rows)

    # Plot all points in a single figure
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.axvline(0.5, color="#cccccc", ls="--", lw=0.8)
    ax.axhline(0.0, color="#cccccc", ls="--", lw=0.8)
    ax.set_xlim(0, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel("Importance (composite)")
    ax.set_ylabel("Valence score (pos_share − neg_share)")
    ax.set_title("Keyword correlometer across FMNs")

    # Draw by group to vary marker, color by aura
    for group in group_names:
        sub = df[df["group"] == group]
        if sub.empty:
            continue
        ax.scatter(
            sub["importance"], sub["valence_score"],
            s=70,
            c=sub["color"],
            marker=marker_map[group],
            edgecolor="black",
            linewidth=0.5,
            alpha=0.95,
        )

    # Optional annotations
    if annotate:
        for _, r in df.iterrows():
            ax.annotate(
                r["keyword"],
                (r["importance"], r["valence_score"]),
                textcoords="offset points",
                xytext=(6, 5),
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.65),
                zorder=5,
            )

    # Legends
    # 1) Small valence legend inside axes (doesn't obstruct much)
    aura_handles = [
        Line2D([0], [0], marker='o', color='w', label=lab,
               markerfacecolor=col, markeredgecolor='black', markersize=8)
        for lab, col in {"positive": "#1f77b4", "neutral": "#7f7f7f", "negative": "#d62728"}.items()
    ]
    leg_val = ax.legend(handles=aura_handles, title="Valence",
                        loc="lower right", framealpha=0.9, fontsize=8, title_fontsize=9)
    ax.add_artist(leg_val)

    # 2) Big "FMN group" legend outside at the bottom, in multiple columns
    marker_handles = [
        Line2D([0], [0], marker=marker_map[g], color='w', label=g,
               markerfacecolor='#aaaaaa', markeredgecolor='black', markersize=8)
        for g in group_names
    ]
    n_groups = max(1, len(group_names))
    # Choose number of columns to keep legend compact
    if n_groups <= 8:
        ncol = 2
    elif n_groups <= 16:
        ncol = 4
    else:
        ncol = 6
    rows_needed = math.ceil(n_groups / ncol)
    # Reserve bottom margin proportional to number of rows in the legend
    bottom_margin = min(0.12 + 0.05 * (rows_needed - 1), 0.35)
    fig.subplots_adjust(bottom=bottom_margin)

    fig.legend(handles=marker_handles, title="FMN group",
               loc="lower center", ncol=ncol, framealpha=0.9,
               fontsize=8, title_fontsize=9)

    # Final styling
    ax.grid(True, alpha=0.15)
    # Keep space for bottom legend
    fig.tight_layout(rect=(0, bottom_margin, 1, 1))

    if missing:
        miss_str = ", ".join([f"{g}:{k}" for g, k in missing[:30]])
        extra = " ..." if len(missing) > 30 else ""
        print(f"[info] {len(missing)} (group, keyword) not found and were skipped: {miss_str}{extra}")

    return {"df": df, "fig": fig, "ax": ax, "missing": missing}


# Bulk correlometer across all FMNs and keywords
bulk = keyword_correlometer_bulk(
    fmns=fmns,
    keywords=keywords,
    emo=emo,
    per_fmn_custom_valences=valences_per_fmn,  # or None to use lexicon
    include_target=False,
    annotate=False,  # was True; disable to reduce clutter
)
display(bulk["df"].head(20))

In [ ]:
# Annotate only extremes on the correlometer (optional)
df = bulk["df"].copy()
ax = bulk["ax"]

K = 8  # number of labels to show from each criterion
top_imp = df.nlargest(K, "importance")
top_abs_val = df.assign(ext=np.abs(df["valence_score"])).nlargest(K, "ext")
to_annot = pd.concat([top_imp, top_abs_val]).drop_duplicates(subset=["group","keyword"])

for _, r in to_annot.iterrows():
    ax.annotate(
        r["keyword"],
        (r["importance"], r["valence_score"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.65),
        zorder=5,
    )
bulk["fig"].canvas.draw_idle()

# Export correlometer figure
corr_fig_path = os.path.join(figures_output_dir, "keyword_correlometer.png")
bulk["fig"].savefig(corr_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved correlometer figure to {corr_fig_path}")

In [ ]:
def keyword_compound_index(
    fmns: dict,
    keywords: list[str],
    emo=emo,
    per_fmn_custom_valences: dict | None = None,
    language: str | None = None,
    include_target: bool = False,
    case_insensitive: bool = True,
    normalize_within_fmn: bool = True,
) -> dict:
    """
    Compute a compound index for each (keyword, FMN) combining:
      - Keyword Relevance Score: closeness centrality (raw and [0,1] scaled per FMN)
      - Connection Count Index: degree (or strength if weighted) (raw and [0,1] scaled per FMN)
      - Aura Valence Mode: mode label of neighbors' valence (positive/neutral/negative)

    compound_index is (closeness_scaled, connection_scaled, aura_label) with the
    two numeric values rounded to 2 decimals.
    """
    import networkx as nx
    import numpy as np
    import pandas as pd

    if language is None:
        language = getattr(emo, "language", "italian")

    def norm(s: str) -> str:
        return str(s).casefold() if case_insensitive else str(s)

    def minmax_scale(d: dict[str, float]) -> dict[str, float]:
        if not d:
            return {}
        vals = np.array(list(d.values()), dtype=float)
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            return {k: 0.0 for k in d}
        rng = (vmax - vmin)
        return {k: (float(v) - float(vmin)) / float(rng) for k, v in d.items()}

    def has_weights(G: nx.Graph) -> bool:
        for _, _, data in G.edges(data=True):
            if isinstance(data, dict) and "weight" in data:
                return True
        return False

    def node_strengths(G: nx.Graph) -> dict:
        strengths = {n: 0.0 for n in G.nodes()}
        if isinstance(G, (nx.MultiGraph, nx.MultiDiGraph)):
            for u, v, data in G.edges(data=True):
                w = float(data.get("weight", 1.0))
                strengths[u] += w
                strengths[v] += w
        else:
            for u, v, data in G.edges(data=True):
                w = float(data.get("weight", 1.0))
                strengths[u] += w
                strengths[v] += w
        return strengths

    rows = []
    missing = []
    connection_kind_per_group = {}

    for group, fmn in fmns.items():
        G = emo.formamentis_to_nxgraph(fmn)

        try:
            closeness = nx.closeness_centrality(G)
        except Exception:
            closeness = {n: 0.0 for n in G.nodes()}

        if has_weights(G):
            connection = node_strengths(G)
            connection_kind_per_group[group] = "strength"
        else:
            deg = dict(G.degree())
            connection = {n: float(deg.get(n, 0)) for n in G.nodes()}
            connection_kind_per_group[group] = "degree"

        if normalize_within_fmn:
            closeness_s = minmax_scale(closeness)
            connection_s = minmax_scale(connection)
        else:
            closeness_s = closeness
            connection_s = connection

        custom_sets = None
        if isinstance(per_fmn_custom_valences, dict):
            custom_sets = per_fmn_custom_valences.get(group, None)

        vertices = list(getattr(fmn, "vertices", G.nodes()))
        vmap_lower = {norm(v): v for v in vertices}

        for kw in keywords:
            kn = norm(kw)
            if kn not in vmap_lower:
                missing.append((group, kw))
                continue
            node = vmap_lower[kn]

            aura = compute_node_aura(
                fmn=fmn,
                target_word=node,
                emo=emo,
                language=language,
                custom_valences=custom_sets,
                include_target=include_target,
                return_details=False,
                case_insensitive=False,
            )

            closeness_scaled_val = float(closeness_s.get(node, 0.0))
            connection_scaled_val = float(connection_s.get(node, 0.0))

            rows.append({
                "group": group,
                "keyword": kw,
                "node": node,
                "closeness": float(closeness.get(node, 0.0)),
                "closeness_scaled": closeness_scaled_val,
                "connection_kind": connection_kind_per_group[group],
                "connection": float(connection.get(node, 0.0)),
                "connection_scaled": connection_scaled_val,
                "aura_label": aura["aura_label"],
                "aura_cat": aura["aura_cat"],
                "pos_share": float(aura["shares"]["positive"]),
                "neu_share": float(aura["shares"]["neutral"]),
                "neg_share": float(aura["shares"]["negative"]),
                "frame_size": int(aura["frame_size"]),
                "considered_size": int(aura["considered_size"]),
                # Rounded to 2 decimals as requested
                "compound_index": (
                    round(closeness_scaled_val, 2),
                    round(connection_scaled_val, 2),
                    aura["aura_label"],
                ),
            })

    df = pd.DataFrame(rows).sort_values(["keyword", "group"]).reset_index(drop=True)

    try:
        wide = pd.concat(
            [
                df.pivot(index="keyword", columns="group", values="closeness_scaled").add_prefix("closeness|"),
                df.pivot(index="keyword", columns="group", values="connection_scaled").add_prefix("connection|"),
                df.pivot(index="keyword", columns="group", values="aura_label").add_prefix("aura|"),
            ],
            axis=1,
        )
    except Exception:
        wide = pd.DataFrame()

    meta = {
        "connection_kind_per_group": connection_kind_per_group,
        "missing": missing,
    }
    if missing:
        miss_str = ", ".join([f"{g}:{k}" for g, k in missing[:30]])
        extra = " ..." if len(missing) > 30 else ""
        print(f"[info] {len(missing)} (group, keyword) not found and were skipped: {miss_str}{extra}")

    return {"df": df, "wide": wide, "meta": meta}


# Compound index for all keywords across all FMNs
result = keyword_compound_index(
    fmns=fmns,
    keywords=keywords,
    emo=emo,
    per_fmn_custom_valences=valences_per_fmn,
)

# Export result["wide"] for reporting
out_csv = os.path.join(base_output_path, project_name, "aggregated", fmn_type, "compound_index_wide.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
result["wide"].to_csv(out_csv)
print(f"Saved: {out_csv}")

display(result["df"].head(20))
display(result["wide"].head())

In [ ]:
df = bulk["df"].copy()

# Faceted correlometer per FMN group
g = sns.relplot(
    data=df,
    x="importance", y="valence_score",
    hue="aura_label",
    col="group", col_wrap=4,
    kind="scatter",
    palette={"positive":"#3b82f6","neutral":"#9ca3af","negative":"#ef4444"},
    edgecolor="black", linewidth=0.4, alpha=0.9,
    height=3.2, aspect=1.05,
)
# Add labels per facet
def _add_labels(data, color=None, **kws):
    ax = plt.gca()
    for _, r in data.iterrows():
        ax.annotate(
            r["keyword"],
            (r["importance"], r["valence_score"]),
            textcoords="offset points",
            xytext=(4, 3),
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.6),
            clip_on=True,
            zorder=6,
        )
g.map_dataframe(_add_labels)

for ax in g.axes.flat:
    ax.axvline(0.5, color="#d1d5db", ls="--", lw=0.8)
    ax.axhline(0.0, color="#d1d5db", ls="--", lw=0.8)
    ax.set_xlim(0,1); ax.set_ylim(-1,1)
g.set_titles(col_template="{col_name}")
g.figure.tight_layout()

# Export faceted figure
facet_fig_path = os.path.join(figures_output_dir, "keyword_correlometer_faceted.png")
g.figure.savefig(facet_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved faceted correlometer figure to {facet_fig_path}")

In [ ]:
# 2) Heatmaps: keywords × groups for valence and importance
df = bulk["df"].copy()
val_pivot = df.pivot(index="keyword", columns="group", values="valence_score").sort_index()
imp_pivot = df.pivot(index="keyword", columns="group", values="importance").sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.35*len(val_pivot))), dpi=130, constrained_layout=True)
sns.heatmap(val_pivot, ax=axes[0], cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.2, linecolor="#eeeeee", cbar_kws={"label":"valence_score"})
axes[0].set_title("Valence score (pos_share − neg_share)")
sns.heatmap(imp_pivot, ax=axes[1], cmap="YlGnBu", vmin=0, vmax=1,
            linewidths=0.2, linecolor="#eeeeee", cbar_kws={"label":"importance"})
axes[1].set_title("Importance (0–1)")
for ax in axes: ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

# Optional: clustered heatmap to reveal blocks
g = sns.clustermap(val_pivot.fillna(0), cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                   figsize=(10,10), metric="euclidean", method="average")
g.figure.suptitle("Clustered valence patterns"); plt.show()

# Export heatmap figure
heatmap_fig_path = os.path.join(figures_output_dir, "keyword_heatmaps.png")
fig.savefig(heatmap_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved heatmap figure to {heatmap_fig_path}")

In [ ]:
# 3) Outliers: most central and most extreme valence
df = bulk["df"].copy()
top_extreme = (
    df.assign(ext=np.abs(df["valence_score"]))
      .sort_values(["importance","ext"], ascending=[False, False])
      .head(20)[["group","keyword","importance","valence_score","aura_label","n_neighbors"]]
)
display(top_extreme)

In [ ]:
# 4) Aura distribution per group (across your keywords)
df = bulk["df"].copy()
counts = df.groupby(["group","aura_label"]).size().unstack(fill_value=0).reindex(columns=["positive","neutral","negative"])
counts = counts.div(counts.sum(axis=1), axis=0)
ax = counts.plot(kind="bar", stacked=True, figsize=(10,4),
                 color={"positive":"#3b82f6","neutral":"#9ca3af","negative":"#ef4444"})
ax.set_ylabel("Share"); ax.set_title("Aura distribution across keywords by group")
plt.tight_layout(); plt.show()

# Export aura distribution figure
aura_dist_fig_path = os.path.join(figures_output_dir, "keyword_aura_distribution.png")
fig = ax.get_figure()
fig.savefig(aura_dist_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved aura distribution figure to {aura_dist_fig_path}")

In [ ]:
# 5) Stable vocabulary across FMNs (from compare_fmns_patterns results)
nodes_df = results["nodes_df"]
stable = (
    nodes_df[(nodes_df["share"]>=0.5) & (nodes_df["dominance_ratio"]>=0.8)]
    .sort_values(["share","dominance_ratio"], ascending=False)
    .head(50)
)
display(stable)

**Connection scaled** is a per-keyword, per-FMN normalized measure of how connected that node is inside that FMN.

How to interpret:

0.0 = least connected (within that FMN), 1.0 = most connected (within that FMN).
It's a relative (within-group) connectivity score, not an absolute network-wide metric.
The companion column connection_kind_per_group tells whether the raw connection is "degree" or "strength".
Notes / caveats:

node_strengths implementation adds each edge weight to both endpoints (undirected interpretation).
If you want a global scale across FMNs, set normalize_within_fmn=False and scale externally.

In [ ]:
# 6) Slopegraph: compare two groups for a metric (from result["df"])
def slope_compare_groups(
    result_df,
    g1,
    g2,
    metric="closeness_scaled",
    top_k=20,
    jitter=True,
    jitter_amount: float | None = None,
    jitter_seed: int | None = 0,
    min_sep_factor: float = 0.04,  # fraction of data-range used as minimum vertical separation
):
    """
    Draw a slopegraph comparing metric for keywords between g1 and g2.
    Uses a deterministic label-stacking algorithm to avoid overlaps.
    jitter_amount (data units) if provided overrides heuristic.
    jitter_seed controls any small random perturbation used for tie-breaking.
    """
    sub = result_df[result_df["group"].isin([g1, g2])]
    piv = sub.pivot(index="keyword", columns="group", values=metric).dropna()
    if piv.empty:
        raise ValueError("No keywords found for the two groups with the requested metric.")
    piv["avg"] = piv[[g1, g2]].mean(axis=1)
    piv = piv.sort_values("avg", ascending=False).head(top_k).drop(columns="avg")

    fig, ax = plt.subplots(figsize=(6, 0.35 * len(piv) + 1), dpi=130)

    # data range and jitter / separation heuristics
    vals = piv[[g1, g2]].values.flatten()
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    data_range = max(1e-6, vmax - vmin)
    if jitter_amount is None:
        jitter_amount = data_range * 0.02

    # deterministic RNG for reproducibility
    rng = np.random.default_rng(jitter_seed) if jitter_seed is not None else np.random.default_rng()

    # draw lines and dots
    for i, (kw, row) in enumerate(piv.iterrows()):
        y1, y2 = float(row[g1]), float(row[g2])
        ax.plot([0, 1], [y1, y2], color="#6b7280", lw=1.5, zorder=1)
        ax.scatter([0, 1], [y1, y2], color=["#2563eb", "#16a34a"], s=30, zorder=3)

    # LABEL PLACEMENT: compute non-overlapping y positions for left-side labels
    left_y = np.array([float(v) for v in piv[g1].values], dtype=float)
    n = len(left_y)
    if n == 0:
        return fig, ax

    # small random tie-break to avoid identical values stacking deterministically
    if jitter:
        left_y = left_y + rng.uniform(-jitter_amount*0.2, jitter_amount*0.2, size=n)

    # minimal separation in data units
    min_sep = max(1e-6, (jitter_amount if jitter_amount is not None else data_range * min_sep_factor))
    # stacking algorithm: forward then backward pass
    order = np.argsort(left_y)
    y_sorted = left_y[order].copy()

    # forward pass
    for i in range(1, n):
        if y_sorted[i] - y_sorted[i - 1] < min_sep:
            y_sorted[i] = y_sorted[i - 1] + min_sep
    # backward pass
    for i in range(n - 2, -1, -1):
        if y_sorted[i + 1] - y_sorted[i] < min_sep:
            y_sorted[i] = y_sorted[i + 1] - min_sep

    # Map back to original order
    adjusted_left_y = np.empty_like(y_sorted)
    adjusted_left_y[order] = y_sorted

    # optionally clamp adjusted positions to data min/max plus a small pad
    pad = max(min_sep, data_range * 0.01)
    global_min, global_max = vmin - pad, vmax + pad
    adjusted_left_y = np.clip(adjusted_left_y, global_min, global_max)

    # Draw left-side labels at adjusted positions
    x_label_pos = -0.02
    for (kw, _), ylab in zip(piv.iterrows(), adjusted_left_y):
        ax.text(x_label_pos, float(ylab), kw, ha="right", va="center", fontsize=8, zorder=4)

    # Also draw right-side short labels near the right points to help readability (optional)
    # keep them aligned to the right point but avoid overlap with left labels by small horizontal offset
    x_right = 1.02
    for (kw, row) in piv.iterrows():
        y2 = float(row[g2])
        ax.text(x_right, y2, "", ha="left", va="center", fontsize=8, zorder=4)  # empty by default

    # ticks / limits
    ax.set_xticks([0, 1])
    ax.set_xticklabels([g1, g2])
    ax.set_xlim(-0.45, 1.45)

    # ensure y-limits include adjusted labels
    y_all = np.concatenate([piv[g1].values, piv[g2].values, adjusted_left_y])
    y_min, y_max = np.nanmin(y_all), np.nanmax(y_all)
    y_pad = max(0.02 * (y_max - y_min if y_max > y_min else 1.0), min_sep)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)

    # set y-axis label to the metric (make it human readable)
    y_label = str(metric).replace("_", " ").title()
    ax.set_ylabel(y_label)

    ax.set_title(f"{metric} comparison")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    return fig, ax

# Usage example
g1 = "highschool_north_low_anxiety_bfmns"
g2 = "highschool_north_high_anxiety_bfmns"
g3 = "psychology_low_anxiety_bfmns"
g4 = "psychology_high_anxiety_bfmns"
g5 = "highschool_south_unfiltered_bfmns"
g6 = "physics_bfmns"
g7 = "researchers_filtered_bfmns"
fig, ax = slope_compare_groups(
    result["df"],
    g1,
    g2,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g1}_vs_{g2}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g3,
    g4,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g3}_vs_{g4}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g3,
    g5,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g3}_vs_{g5}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g5,
    g6,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g5}_vs_{g6}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")